In [1]:
import pandas as pd
import numpy as np

doc_emb_df = pd.read_csv('/config/workspace/datasets/Tasks/3A/antique_doc_embeddings.csv')

# Convert strings back to NumPy vectors
doc_emb_df['doc_embeddings'] = [np.fromstring(vec[1:-1], dtype=float, sep=',') for vec in doc_emb_df['doc_embeddings']]

print(doc_emb_df)

          doc_id                                     doc_embeddings
0      1964316_5  [-0.0132087674, -0.0825422108, 0.0144632952, -...
1     1674088_11  [0.0110482518, 0.09191861, 0.00595060643, 0.00...
2     1218838_13  [-0.0290984362, 0.058977358, -0.0251658596, -0...
3     1519022_15  [-0.0271731708, 0.0373504609, 0.0298253279, -0...
4      3059341_5  [-0.00224227854, -0.0586593598, 0.0215318352, ...
...          ...                                                ...
6481    247023_6  [0.0285549182, -0.0799941644, -0.000722727331,...
6482   1499030_5  [0.0117990598, -0.0142808128, -0.0145873558, -...
6483   2916758_0  [0.0143310865, 0.00705553312, 0.00457704067, -...
6484  1105845_15  [0.0133153796, 0.0152588673, 0.000567730691, 0...
6485   3699008_1  [-0.0214124434, -0.0110130869, 0.0243909713, -...

[6486 rows x 2 columns]


In [3]:
doc_emb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   doc_id          6486 non-null   object
 1   doc_embeddings  6486 non-null   object
dtypes: object(2)
memory usage: 101.5+ KB


In [3]:
import pandas as pd
import numpy as np

df_loaded = pd.read_csv('/config/workspace/datasets/Tasks/3A/antique_train_queries.csv')

# Convert strings back to NumPy vectors
df_loaded['query_embeddings'] = [np.fromstring(vec[1:-1], dtype=float, sep=',') for vec in df_loaded['query_embeddings']]

print(df_loaded)

      query_id                                   query_embeddings      doc_id  \
0      1964316  [0.00820141565, 0.0182990301, 0.0174273867, -0...   1964316_5   
1      1964316  [0.00820141565, 0.0182990301, 0.0174273867, -0...  1674088_11   
2      1964316  [0.00820141565, 0.0182990301, 0.0174273867, -0...  1218838_13   
3      1964316  [0.00820141565, 0.0182990301, 0.0174273867, -0...  1519022_15   
4      1964316  [0.00820141565, 0.0182990301, 0.0174273867, -0...   3059341_5   
...        ...                                                ...         ...   
1646   1957887  [0.074257873, 0.00764932111, 0.0062787463, 0.0...   4375962_5   
1647   1957887  [0.074257873, 0.00764932111, 0.0062787463, 0.0...   698757_23   
1648   1957887  [0.074257873, 0.00764932111, 0.0062787463, 0.0...   2860539_1   
1649   1957887  [0.074257873, 0.00764932111, 0.0062787463, 0.0...   2434074_1   
1650   1957887  [0.074257873, 0.00764932111, 0.0062787463, 0.0...   3562931_7   

      relevance  
0        

In [4]:
"""
QuantumCLEF 2025 — Task 3A
QUBO k-Medoids Clustering + Retrieval
=====================================

Features:
- Uses qclef qa.submit() for BOTH SA and QA
- Global query-aware k-medoids
- Diversity-aware QUBO
- Davies-Bouldin evaluation
- nDCG@10 evaluation
- Automatic submission file generation
- Saves files in /config/workspace/submissions

Run twice:
    USE_QPU = False  -> SA submission
    USE_QPU = True   -> QA submission
"""

import os
import ast
import uuid
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import davies_bouldin_score

import dimod
from dimod import BinaryQuadraticModel
from dimod.generators import combinations

from neal import SimulatedAnnealingSampler

from dwave.system import (
    DWaveSampler,
    EmbeddingComposite
)
import time

from dwave.preprocessing import ScaleComposite

from qclef import qa_access as qa

# =========================================================
# CONFIG
# =========================================================

DOC_PATH = (
    "/config/workspace/datasets/Tasks/3A/"
    "antique_doc_embeddings.csv"
)

QUERY_PATH = (
    "/config/workspace/datasets/Tasks/3A/"
    "antique_train_queries.csv"
)

SUBMIT_PATH = "/config/workspace/submissions"

GROUP_NAME = "FAST-NUCES"

METHOD_ID = "KMEDOIDS_QA_1000"

USE_QPU = False

METHOD = "QA" if USE_QPU else "SA"

K_VALUES = [10,25,50]

# =========================================================
# QUBO PARAMS
# =========================================================

BATCH_SIZE = 250

KNN_K = 30

ALPHA = 2.5
BETA = 0.7
GAMMA = 1.5

CONSTRAINT = 20.0

SA_NUM_READS = 1000

QPU_NUM_READS = 1
CHAIN_STRENGTH = 5.0

np.random.seed(42)

# =========================================================
# LOAD DATA
# =========================================================

def parse_vec(x):

    if isinstance(x, str):

        return np.array(
            ast.literal_eval(x),
            dtype=np.float32
        )

    return np.array(
        x,
        dtype=np.float32
    )


def load_docs():

    df = pd.read_csv(DOC_PATH)

    X = np.vstack(
        df["doc_embeddings"]
        .apply(parse_vec)
        .values
    )

    doc_ids = df["doc_id"].values

    X = normalize(X)

    return X, doc_ids


def load_queries():

    df = pd.read_csv(QUERY_PATH)

    df["query_embeddings"] = (
        df["query_embeddings"]
        .apply(parse_vec)
    )

    return df

# =========================================================
# QUERY SIGNAL
# =========================================================

def build_query_signal(
    X,
    query_df
):

    qvecs = np.vstack(
        query_df["query_embeddings"]
        .values
    )

    sims = cosine_similarity(
        X,
        qvecs
    )

    # mean top-5 similarity
    topk = np.sort(
        sims,
        axis=1
    )[:, -5:]

    signal = np.mean(
        topk,
        axis=1
    )

    signal = (
        signal - signal.min()
    ) / (
        signal.max()
        - signal.min()
        + 1e-9
    )

    return signal

# =========================================================
# DENSITY
# =========================================================

def compute_density(X):

    nn = NearestNeighbors(
        n_neighbors=KNN_K,
        metric="cosine"
    )

    nn.fit(X)

    dist, idx = nn.kneighbors(X)

    density = np.zeros(len(X))

    for i in range(len(X)):

        sims = 1.0 - dist[i][1:]

        density[i] = np.mean(sims)

    density = (
        density - density.min()
    ) / (
        density.max()
        - density.min()
        + 1e-9
    )

    return density

# =========================================================
# BATCHING
# =========================================================

def create_batches(n):

    idx = np.arange(n)

    np.random.shuffle(idx)

    return [
        idx[i:i + BATCH_SIZE]
        for i in range(
            0,
            n,
            BATCH_SIZE
        )
    ]

# =========================================================
# BUILD QUBO
# =========================================================

def build_qubo(
    Xb,
    signal_b,
    density_b,
    k_local
):

    n = len(Xb)

    nn = NearestNeighbors(
        n_neighbors=min(
            KNN_K,
            len(Xb)
        ),
        metric="cosine"
    )

    nn.fit(Xb)

    dist, idx = nn.kneighbors(Xb)

    Q = {}

    # -----------------------------------------------------
    # Diagonal terms
    # -----------------------------------------------------

    for i in range(n):

        score = (
            ALPHA * signal_b[i]
            + GAMMA * density_b[i]
        )

        Q[(i, i)] = -score

    # -----------------------------------------------------
    # Diversity penalty
    # -----------------------------------------------------

    for i in range(n):

        for j, d in zip(
            idx[i][1:],
            dist[i][1:]
        ):

            if i < j:

                sim = 1.0 - d

                Q[(i, j)] = (
                    Q.get((i, j), 0)
                    + BETA * sim
                )

    # -----------------------------------------------------
    # Build BQM
    # -----------------------------------------------------

    bqm = BinaryQuadraticModel.from_qubo(Q)

    bqm.update(
        combinations(
            range(n),
            k_local,
            strength=CONSTRAINT
        )
    )

    return bqm

# =========================================================
# QCLEF SUBMIT
# =========================================================

def submit_sa(
    bqm,
    batch_id,
    k
):

    sampler = SimulatedAnnealingSampler()

    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        bqm,
        label=(
            f"3A_SA_k{k}_"
            f"b{batch_id}_"
            f"{METHOD_ID}"
        ),
        num_reads=SA_NUM_READS
    )

    best = sampleset.first.sample

    pid = sampleset.info.get(
        "problem_id",
        str(uuid.uuid4())
    )

    return best, pid


def submit_qpu(
    bqm,
    batch_id,
    k
):

    sampler = EmbeddingComposite(
        ScaleComposite(
            DWaveSampler()
        )
    )

    sampleset = qa.submit(
        sampler,
        EmbeddingComposite.sample,
        bqm,
        label=(
            f"3A_QPU_k{k}_"
            f"b{batch_id}_"
            f"{METHOD_ID}"
        ),
        num_reads=QPU_NUM_READS,
        chain_strength=CHAIN_STRENGTH
    )

    best = sampleset.first.sample

    pid = sampleset.info.get(
        "problem_id",
        str(uuid.uuid4())
    )

    return best, pid

# =========================================================
# SOLVE BATCHES
# =========================================================

def solve_kmedoids(
    X,
    signal,
    density,
    k
):

    n = len(X)

    batches = create_batches(n)

    selected = []

    problem_ids = []

    solver_tag = (
        "QPU"
        if USE_QPU
        else "SA"
    )

    print(
        f"\n{k} medoids | "
        f"{len(batches)} batches | "
        f"solver={solver_tag}"
    )

    for b_idx, batch in enumerate(batches):

        print(
            f"\nBatch "
            f"{b_idx+1}/"
            f"{len(batches)}"
        )

        Xb = X[batch]

        signal_b = signal[batch]

        density_b = density[batch]

        k_local = max(
            1,
            int(
                k * len(batch) / n
            )
        )

        bqm = build_qubo(
            Xb,
            signal_b,
            density_b,
            k_local
        )

        try:

            if USE_QPU:

                best, pid = submit_qpu(
                    bqm,
                    b_idx,
                    k
                )

            else:

                best, pid = submit_sa(
                    bqm,
                    b_idx,
                    k
                )

        except Exception as e:

            print(
                f"Submission failed: {e}"
            )

            print(
                "Falling back to local SA"
            )

            ss = (
                SimulatedAnnealingSampler()
                .sample(
                    bqm,
                    num_reads=SA_NUM_READS
                )
            )

            best = ss.first.sample

            pid = str(uuid.uuid4())

        chosen = [
            i for i, v in best.items()
            if v == 1
        ]

        selected.extend(
            batch[chosen]
        )

        problem_ids.append(pid)

        print(
            f"Selected "
            f"{len(chosen)} docs"
        )

    # -----------------------------------------------------
    # unique medoids
    # -----------------------------------------------------

    selected = list(set(selected))

    # -----------------------------------------------------
    # trim if needed
    # -----------------------------------------------------

    if len(selected) > k:

        selected = selected[:k]

    centroid_vectors = X[selected]

    return (
        centroid_vectors,
        selected,
        problem_ids
    )

# =========================================================
# ASSIGN CLUSTERS
# =========================================================

def assign_clusters(
    X,
    centroid_vectors,
    centroid_ids,
    doc_ids
):

    nn = NearestNeighbors(
        n_neighbors=1,
        metric="cosine"
    )

    nn.fit(centroid_vectors)

    _, idx = nn.kneighbors(X)

    clusters = {
        int(cid): []
        for cid in centroid_ids
    }

    labels = np.zeros(
        len(X),
        dtype=int
    )

    for i, c in enumerate(idx.flatten()):

        centroid = centroid_ids[c]

        clusters[int(centroid)].append(
            doc_ids[i]
        )

        labels[i] = c

    return clusters, labels

# =========================================================
# nDCG
# =========================================================

def dcg(rels):

    return sum(
        r / np.log2(i + 2)
        for i, r in enumerate(rels)
    )


def ndcg_at_10(rels):

    ideal = sorted(
        rels,
        reverse=True
    )

    return dcg(rels) / (
        dcg(ideal) + 1e-9
    )

# =========================================================
# EVALUATE RETRIEVAL
# =========================================================

def evaluate_ndcg(
    X,
    clusters,
    query_df,
    centroid_ids,
    doc_ids
):

    nn = NearestNeighbors(
        n_neighbors=1,
        metric="cosine"
    )

    nn.fit(X[centroid_ids])

    scores = []

    doc_map = {
        d: i
        for i, d in enumerate(doc_ids)
    }

    for qid, group in query_df.groupby("query_id"):

        qvec = group[
            "query_embeddings"
        ].iloc[0]

        # nearest centroid
        c_idx = nn.kneighbors(
            [qvec],
            return_distance=False
        )[0][0]

        centroid = centroid_ids[c_idx]

        cluster_docs = clusters[centroid]

        cluster_idx = [
            doc_map[d]
            for d in cluster_docs
        ]

        cluster_vecs = X[cluster_idx]

        # rerank docs
        sims = cosine_similarity(
            [qvec],
            cluster_vecs
        )[0]

        ranked = sorted(
            zip(cluster_docs, sims),
            key=lambda x: x[1],
            reverse=True
        )

        ranked_docs = [
            d for d, _ in ranked
        ]

        rel_map = {
            row["doc_id"]: row["relevance"]
            for _, row in group.iterrows()
        }

        rels = [
            rel_map.get(d, 0)
            for d in ranked_docs[:10]
        ]

        scores.append(
            ndcg_at_10(rels)
        )

    return np.mean(scores)

# =========================================================
# DB INDEX
# =========================================================

def evaluate_dbi(
    X,
    labels
):

    return davies_bouldin_score(
        X,
        labels
    )

# =========================================================
# WRITE SUBMISSION
# =========================================================

def write_submission(
    k,
    centroid_vectors,
    centroid_ids,
    clusters,
    problem_ids
):

    os.makedirs(
        SUBMIT_PATH,
        exist_ok=True
    )

    filename = (
        f"{k}_{METHOD}_"
        f"{GROUP_NAME}_"
        f"{METHOD_ID}.txt"
    )

    path = os.path.join(
        SUBMIT_PATH,
        filename
    )

    lines = []

    for i, cid in enumerate(centroid_ids):

        item = {
            "centroid": (
                centroid_vectors[i]
                .tolist()
            ),
            "docs": list(
                clusters[cid]
            )
        }

        lines.append(str(item))

    # last line = problem ids
    lines.append(
        str(problem_ids)
    )

    with open(path, "w") as f:

        f.write(
            "\n".join(lines)
        )

    print(
        f"\nSaved submission:"
    )

    print(path)

# =========================================================
# MAIN
# =========================================================

def main():

    print("\nLoading documents...", flush=True)

    X, doc_ids = load_docs()

    print("Loading queries...", flush=True)

    query_df = load_queries()

    print("\nBuilding query signal...", flush=True)

    signal = build_query_signal(
        X,
        query_df
    )

    print("Computing density...", flush=True)

    density = compute_density(X)

    # =====================================================
    # RUN FOR EACH K
    # =====================================================

    for k in K_VALUES:

        print("\n" + "=" * 60, flush=True)
        print(f"RUNNING k = {k}", flush=True)
        print("=" * 60, flush=True)

        t0 = time.time()

        # -------------------------------------------------
        # Solve QUBO
        # -------------------------------------------------

        try:

            (
                centroid_vectors,
                centroid_ids,
                problem_ids
            ) = solve_kmedoids(
                X,
                signal,
                density,
                k
            )

            print(
                f"\nSelected centroids: {len(centroid_ids)}",
                flush=True
            )

        except Exception as e:

            print(
                f"\nERROR during solve_kmedoids(): {e}",
                flush=True
            )

            continue

        # -------------------------------------------------
        # Assign clusters
        # -------------------------------------------------

        try:

            clusters, labels = assign_clusters(
                X,
                centroid_vectors,
                centroid_ids,
                doc_ids
            )

            print(
                "Cluster assignment completed.",
                flush=True
            )

        except Exception as e:

            print(
                f"\nERROR during cluster assignment: {e}",
                flush=True
            )

            continue

        # -------------------------------------------------
        # Davies-Bouldin Index
        # -------------------------------------------------

        try:

            dbi = evaluate_dbi(
                X,
                labels
            )

            print(
                f"\nDavies-Bouldin Index : {dbi:.6f}",
                flush=True
            )

        except Exception as e:

            print(
                f"\nERROR computing DB Index: {e}",
                flush=True
            )

            dbi = None

        # -------------------------------------------------
        # nDCG@10
        # -------------------------------------------------

        try:

            ndcg = evaluate_ndcg(
                X,
                clusters,
                query_df,
                centroid_ids,
                doc_ids
            )

            print(
                f"nDCG@10              : {ndcg:.6f}",
                flush=True
            )

        except Exception as e:

            print(
                f"\nERROR computing nDCG: {e}",
                flush=True
            )

            ndcg = None

        # -------------------------------------------------
        # Save submission
        # -------------------------------------------------

        try:

            write_submission(
                k,
                centroid_vectors,
                centroid_ids,
                clusters,
                problem_ids
            )

        except Exception as e:

            print(
                f"\nERROR writing submission: {e}",
                flush=True
            )

        print(
            f"\nFinished k={k} "
            f"in {time.time()-t0:.2f}s",
            flush=True
        )

    print("\nDONE ✔", flush=True)


# =========================================================
# ENTRY
# =========================================================

if __name__ == "__main__":

    main()


Loading documents...
Loading queries...

Building query signal...
Computing density...

RUNNING k = 10

10 medoids | 26 batches | solver=SA

Batch 1/26
Selected 1 docs

Batch 2/26
Selected 1 docs

Batch 3/26
Selected 1 docs

Batch 4/26
Selected 1 docs

Batch 5/26
Selected 1 docs

Batch 6/26
Selected 1 docs

Batch 7/26
Selected 1 docs

Batch 8/26
Selected 1 docs

Batch 9/26
Selected 1 docs

Batch 10/26
Selected 1 docs

Batch 11/26
Selected 1 docs

Batch 12/26
Selected 1 docs

Batch 13/26
Selected 1 docs

Batch 14/26
Selected 1 docs

Batch 15/26
Selected 1 docs

Batch 16/26
Selected 1 docs

Batch 17/26
Selected 1 docs

Batch 18/26
Selected 1 docs

Batch 19/26
Selected 1 docs

Batch 20/26
Selected 1 docs

Batch 21/26
Selected 1 docs

Batch 22/26
Selected 1 docs

Batch 23/26
Selected 1 docs

Batch 24/26
Selected 1 docs

Batch 25/26
Selected 1 docs

Batch 26/26
Selected 1 docs

Selected centroids: 10
Cluster assignment completed.

Davies-Bouldin Index : 7.536601
nDCG@10              : 0.88

# last try

In [2]:

import os
import ast
import uuid
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import davies_bouldin_score

import dimod
from dimod import BinaryQuadraticModel
from dimod.generators import combinations

from neal import SimulatedAnnealingSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.preprocessing import ScaleComposite
from qclef import qa_access as qa

warnings.filterwarnings("ignore")

DOC_PATH   = "/config/workspace/datasets/Tasks/3A/antique_doc_embeddings.csv"
QUERY_PATH = "/config/workspace/datasets/Tasks/3A/antique_train_queries.csv"
SUBMIT_PATH = "/config/workspace/submissions"

GROUP_NAME = "FAST-NUCES"
METHOD_ID  = "FPS-QUERYAWARE"    # same for SA and QPU runs

USE_QPU = True                      # False = SA | True = QPU
METHOD  = "QA" if USE_QPU else "SA"

K_VALUES = [10, 25, 50]

# ── FPS pivot reduction ───────────────────────────────────
N_PIVOTS = 150      # number of FPS pivots → must fit QPU (~150 max)
                    # increase to 200-300 for SA-only runs

# ── Composite score weights ───────────────────────────────
ALPHA = 2.5         # query_relevance weight (most important for nDCG)
BETA  = 1.5         # density weight         (important for DBI)
GAMMA = 1.0         # coverage weight        (how many docs a pivot represents)

# ── QUBO off-diagonal ─────────────────────────────────────
DELTA  = 0.7        # diversity penalty strength
KNN_K  = 20         # neighbours for off-diagonal (sparse QUBO)

# ── Solver ────────────────────────────────────────────────
SA_NUM_READS   = 1000
QPU_NUM_READS  = 1
CHAIN_STRENGTH = 5.0

np.random.seed(42)


# =========================================================
# 1. DATA LOADING
# =========================================================

def parse_vec(x):
    """Parse embedding stored as string list in CSV."""
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=np.float32)
    return np.array(x, dtype=np.float32)


def load_docs():
    """Load document embeddings. Returns X [n_docs, 768], doc_ids."""
    print("Loading documents...")
    df     = pd.read_csv(DOC_PATH)
    X      = np.vstack(df["doc_embeddings"].apply(parse_vec).values)
    X      = normalize(X)                     # L2 normalise
    doc_ids = df["doc_id"].values
    print(f"  Docs: {X.shape[0]} × {X.shape[1]}")
    return X, doc_ids


def load_queries():
    """
    Load query embeddings + relevance labels.
    Expected columns: query_id, doc_id, relevance, query_embeddings
    """
    print("Loading queries...")
    df = pd.read_csv(QUERY_PATH)
    df["query_embeddings"] = df["query_embeddings"].apply(parse_vec)
    n_queries = df["query_id"].nunique()
    print(f"  Queries: {n_queries} | Relevance judgements: {len(df)}")
    return df


# =========================================================
# 2. COMPOSITE SCORE
# =========================================================

def build_query_relevance(X, query_df):
    """
    For each document, compute a relevance-weighted query signal.

    Improvement over original:
      Original: mean top-5 cosine_sim(doc, all_queries)  — ignores labels
      Ours    : Σ_j relevance[j] × cosine_sim(doc, query_j)  — label-weighted

    This means a doc near a highly-relevant query gets a much higher
    score than one near a low-relevance query. Directly optimises nDCG@10.
    """
    print("  Computing relevance-weighted query signal...")

    # Get unique query vectors and their mean relevance weight
    query_vecs   = []
    query_weights = []

    for qid, group in query_df.groupby("query_id"):
        qvec   = group["query_embeddings"].iloc[0]
        # mean relevance across all judged docs for this query
        weight = group["relevance"].mean()
        query_vecs.append(qvec)
        query_weights.append(weight)

    Q_mat   = np.vstack(query_vecs)           # [n_queries, 768]
    weights = np.array(query_weights)          # [n_queries]

    # cosine similarity: [n_docs, n_queries]
    sims = cosine_similarity(X, Q_mat)

    # weight by relevance: docs near high-relevance queries score higher
    query_rel = (sims * weights[np.newaxis, :]).sum(axis=1)

    # normalise to [0, 1]
    query_rel = (query_rel - query_rel.min()) / (query_rel.max() - query_rel.min() + 1e-9)

    print(f"    Query relevance: mean={query_rel.mean():.3f}  max={query_rel.max():.3f}")
    return query_rel


def compute_density(X):
    """
    Local density: mean cosine similarity to KNN_K nearest neighbours.
    High density = doc sits in a dense region = good centroid candidate.
    """
    print("  Computing local density...")
    nn = NearestNeighbors(n_neighbors=KNN_K + 1, metric="cosine", n_jobs=-1)
    nn.fit(X)
    dist, _ = nn.kneighbors(X)
    density  = 1.0 - np.mean(dist[:, 1:], axis=1)   # cosine dist → similarity
    density  = (density - density.min()) / (density.max() - density.min() + 1e-9)
    print(f"    Density: mean={density.mean():.3f}  max={density.max():.3f}")
    return density


def composite_score(query_rel, density):
    """
    Combined quality score per document.
    Coverage is added later (after FPS) since it depends on pivot selection.
    """
    score = ALPHA * query_rel + BETA * density
    score = (score - score.min()) / (score.max() - score.min() + 1e-9)
    return score


# =========================================================
# 3. FPS PIVOT REDUCTION  ← key improvement
# =========================================================

def fps_pivots(X, score, n_pivots):
    """
    Query-biased Farthest Point Sampling.

    Standard FPS: greedily select the point most distant from all
    already-selected pivots. Guarantees global spatial coverage.

    Our modification: weight the distance by the composite score so
    high-relevance docs in sparse regions get selected first.

        next_pivot = argmax( min_dist_to_selected × (1 + score) )

    This biases the pivot set toward query-relevant regions while
    still maintaining global coverage — better nDCG without hurting DBI.

    Returns array of n_pivots global indices into X.
    """
    print(f"\n  Running query-biased FPS: {len(X)} docs → {n_pivots} pivots...")
    t0 = time.time()

    n          = len(X)
    selected   = []
    min_dists  = np.full(n, np.inf)

    # seed: highest composite score doc
    first = int(np.argmax(score))
    selected.append(first)

    for step in range(1, n_pivots):
        # update min distances to nearest selected pivot
        last_vec  = X[selected[-1]].reshape(1, -1)
        new_dists = 1.0 - cosine_similarity(X, last_vec).flatten()  # cosine dist
        min_dists = np.minimum(min_dists, new_dists)

        # bias toward high-score docs in sparse regions
        biased = min_dists * (1.0 + score)
        biased[selected] = -np.inf     # don't re-select

        next_pivot = int(np.argmax(biased))
        selected.append(next_pivot)

        if (step + 1) % 30 == 0:
            print(f"    FPS step {step+1}/{n_pivots}...")

    print(f"    FPS done: {len(selected)} pivots in {time.time()-t0:.1f}s")
    return np.array(selected)


def compute_coverage(X, pivot_indices):
    """
    For each pivot, count how many docs it is the nearest pivot to.
    High coverage = this pivot represents many documents = important centroid.
    Normalised to [0, 1].
    """
    nn = NearestNeighbors(n_neighbors=1, metric="cosine", n_jobs=-1)
    nn.fit(X[pivot_indices])
    _, idx = nn.kneighbors(X)
    counts = np.bincount(idx.flatten(), minlength=len(pivot_indices))
    cov    = counts.astype(float)
    cov    = (cov - cov.min()) / (cov.max() - cov.min() + 1e-9)
    return cov


# =========================================================
# 4. BUILD GLOBAL QUBO ON PIVOTS
# =========================================================

def build_global_qubo(X_pivots, pivot_scores, k):
    """
    Build ONE global QUBO on the N_PIVOTS pivot embeddings.

    This replaces the original 26-batch approach with a single
    globally-aware optimisation — all pivots compete simultaneously
    so the result is globally diverse, not batch-locally diverse.

    Q[i,i]  = −pivot_scores[i]              (reward high-quality pivots)
    Q[i,j]  = +DELTA · cosine_sim(i,j)      (penalise redundant pivot pairs)
    Constraint: exactly k pivots selected    (auto-tuned strength)
    """
    n  = len(X_pivots)
    Q  = {}

    # ── Diagonal ──────────────────────────────────────────
    for i in range(n):
        Q[(i, i)] = -float(pivot_scores[i])

    # ── Off-diagonal: sparse kNN similarity ───────────────
    nn = NearestNeighbors(n_neighbors=min(KNN_K, n - 1),
                          metric="cosine", n_jobs=-1)
    nn.fit(X_pivots)
    dist, idx = nn.kneighbors(X_pivots)

    n_edges = 0
    for i in range(n):
        for rank in range(1, min(KNN_K, n - 1)):
            j   = int(idx[i, rank])
            if i >= j:
                continue
            sim = max(0.0, 1.0 - float(dist[i, rank]))
            Q[(i, j)] = Q.get((i, j), 0.0) + DELTA * sim
            n_edges += 1

    # ── BQM + auto-tuned cardinality constraint ────────────
    bqm     = BinaryQuadraticModel.from_qubo(Q)
    penalty = bqm.maximum_energy_delta()       # auto-tunes, not hardcoded 20.0
    bqm.update(combinations(range(n), k, strength=penalty))

    print(f"    QUBO: {n} vars | {n_edges} off-diag edges | "
          f"constraint strength={penalty:.2f}")
    return bqm


# =========================================================
# 5. SOLVERS via qclef
# =========================================================

def submit_sa(bqm, k):
    sampler   = SimulatedAnnealingSampler()
    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        bqm,
        label=f"3A_SA_k{k}_{METHOD_ID}",
        num_reads=SA_NUM_READS,
    )
    best = sampleset.first.sample
    pid  = sampleset.info.get("problem_id", str(uuid.uuid4()))
    print(f"    SA energy={sampleset.first.energy:.4f}")
    return best, pid


def submit_qpu(bqm, k):
    sampler   = EmbeddingComposite(ScaleComposite(DWaveSampler()))
    sampleset = qa.submit(
        sampler,
        EmbeddingComposite.sample,
        bqm,
        label=f"3A_QPU_k{k}_{METHOD_ID}",
        num_reads=QPU_NUM_READS,
        chain_strength=CHAIN_STRENGTH,
    )
    best = sampleset.first.sample
    pid  = sampleset.info.get("problem_id", str(uuid.uuid4()))
    return best, pid


# =========================================================
# 6. SOLVE: FPS + QUBO + SCORE-BASED TOP-K
# =========================================================

def solve_kmedoids(X, pivot_indices, pivot_scores, k):
    """
    Build and solve global QUBO on pivots.
    If QUBO selects wrong number, fall back to score-ranked top-k.

    Returns (centroid_vectors, centroid_global_indices, [problem_id])
    """
    X_pivots = X[pivot_indices]

    print(f"\n  Building QUBO for k={k}...")
    bqm = build_global_qubo(X_pivots, pivot_scores, k)

    print(f"  Solving via {'QPU' if USE_QPU else 'SA'}...")
    t0 = time.time()

    try:
        if USE_QPU:
            best, pid = submit_qpu(bqm, k)
        else:
            best, pid = submit_sa(bqm, k)
    except Exception as e:
        print(f"  Submit failed ({e}) → local SA fallback")
        ss   = SimulatedAnnealingSampler().sample(bqm, num_reads=SA_NUM_READS)
        best = ss.first.sample
        pid  = str(uuid.uuid4())

    print(f"  Solved in {time.time()-t0:.1f}s | pid={pid[:8]}...")

    # Local pivot indices selected by QUBO
    local_selected = [i for i, v in best.items() if v == 1]

    # ── Score-based top-k fix ──────────────────────────────
    # If QUBO returned wrong count, re-rank ALL pivots by score
    # and take top-k (not arbitrary order as in original code)
    if len(local_selected) != k:
        print(f"  QUBO returned {len(local_selected)} ≠ {k} → score-based correction")
        ranked = np.argsort(pivot_scores)[::-1]   # descending score
        # prefer QUBO-selected first, then fill from score ranking
        qubo_set  = set(local_selected)
        final = [i for i in ranked if i in qubo_set]
        extra = [i for i in ranked if i not in qubo_set]
        local_selected = (final + extra)[:k]

    # Map local pivot indices → global doc indices
    global_selected       = pivot_indices[local_selected]
    centroid_vectors      = X[global_selected]

    print(f"  Final centroids: {len(global_selected)}")
    return centroid_vectors, global_selected, [pid]


# =========================================================
# 7. CLUSTER ASSIGNMENT
# =========================================================

def assign_clusters(X, centroid_vectors, centroid_ids, doc_ids):
    """Assign every doc to its nearest centroid by cosine similarity."""
    nn = NearestNeighbors(n_neighbors=1, metric="cosine", n_jobs=-1)
    nn.fit(centroid_vectors)
    _, idx = nn.kneighbors(X)

    clusters = {int(cid): [] for cid in centroid_ids}
    labels   = np.zeros(len(X), dtype=int)

    for i, c in enumerate(idx.flatten()):
        centroid = centroid_ids[c]
        clusters[int(centroid)].append(doc_ids[i])
        labels[i] = c

    sizes = [len(v) for v in clusters.values()]
    print(f"  Cluster sizes: min={min(sizes)} max={max(sizes)} mean={np.mean(sizes):.0f}")
    return clusters, labels


# =========================================================
# 8. EVALUATION
# =========================================================

def evaluate_dbi(X, labels):
    """Davies-Bouldin Index — lower is better."""
    return davies_bouldin_score(X, labels)


def dcg(rels):
    return sum(r / np.log2(i + 2) for i, r in enumerate(rels))


def ndcg_at_10(rels):
    ideal = sorted(rels, reverse=True)
    return dcg(rels[:10]) / (dcg(ideal[:10]) + 1e-9)


def evaluate_ndcg(X, clusters, query_df, centroid_ids, doc_ids):
    """
    nDCG@10 across all 50 queries.
    For each query: find nearest centroid → retrieve cluster docs
    → rank by cosine sim → measure nDCG@10 vs relevance labels.
    """
    nn      = NearestNeighbors(n_neighbors=1, metric="cosine", n_jobs=-1)
    nn.fit(X[centroid_ids])
    doc_map = {d: i for i, d in enumerate(doc_ids)}
    scores  = []

    for qid, group in query_df.groupby("query_id"):
        qvec    = group["query_embeddings"].iloc[0]
        c_idx   = nn.kneighbors([qvec], return_distance=False)[0][0]
        centroid = centroid_ids[c_idx]

        cluster_docs = clusters[centroid]
        if not cluster_docs:
            scores.append(0.0)
            continue

        cluster_idx  = [doc_map[d] for d in cluster_docs if d in doc_map]
        cluster_vecs = X[cluster_idx]
        sims         = cosine_similarity([qvec], cluster_vecs)[0]
        ranked_docs  = [cluster_docs[i] for i in np.argsort(sims)[::-1]]

        rel_map = {row["doc_id"]: row["relevance"] for _, row in group.iterrows()}
        rels    = [rel_map.get(d, 0) for d in ranked_docs[:10]]
        scores.append(ndcg_at_10(rels))

    return float(np.mean(scores))


# =========================================================
# 9. SUBMISSION FILE
# =========================================================

def write_submission(k, centroid_vectors, centroid_ids, clusters, problem_ids):
    """
    Filename: {k}_{SA/QA}_{GROUP}_{METHOD_ID}.txt
    Format  : one line per centroid with vector + doc list
              last line = list of problem UUIDs
    """
    os.makedirs(SUBMIT_PATH, exist_ok=True)
    filename = f"{k}_{METHOD}_{GROUP_NAME}_{METHOD_ID}.txt"
    path     = os.path.join(SUBMIT_PATH, filename)

    lines = []
    for i, cid in enumerate(centroid_ids):
        item = {
            "centroid": centroid_vectors[i].tolist(),
            "docs":     list(clusters[int(cid)])
        }
        lines.append(str(item))
    lines.append(str(problem_ids))

    with open(path, "w") as f:
        f.write("\n".join(lines))

    print(f"  Submission saved → {path}")
    return path


# =========================================================
# 10. MAIN
# =========================================================

def main():
    print(f"\n{'='*60}")
    print(f"Task 3A | {METHOD} | {METHOD_ID}")
    print(f"k values: {K_VALUES} | FPS pivots: {N_PIVOTS}")
    print(f"{'='*60}")

    # ── Load ──────────────────────────────────────────────
    X, doc_ids = load_docs()
    query_df   = load_queries()
    n          = len(X)

    # ── Compute signals on all docs (once) ────────────────
    print("\n[Signals]")
    query_rel = build_query_relevance(X, query_df)
    density   = compute_density(X)
    score     = composite_score(query_rel, density)

    # ── FPS pivot reduction (once for all k values) ────────
    print("\n[FPS Pivot Reduction]")
    pivot_indices = fps_pivots(X, score, N_PIVOTS)

    # Coverage score for pivots
    coverage      = compute_coverage(X, pivot_indices)

    # Final pivot scores (add coverage now that we have pivots)
    pivot_scores  = (
        ALPHA * query_rel[pivot_indices]
      + BETA  * density[pivot_indices]
      + GAMMA * coverage
    )
    pivot_scores  = (pivot_scores - pivot_scores.min()) / (
                     pivot_scores.max() - pivot_scores.min() + 1e-9)

    print(f"  Pivot score: mean={pivot_scores.mean():.3f}  max={pivot_scores.max():.3f}")

    # ── Run for each k ────────────────────────────────────
    for k in K_VALUES:
        print(f"\n{'='*60}")
        print(f"k = {k}")
        print(f"{'='*60}")
        t0 = time.time()

        # Solve QUBO
        try:
            centroid_vectors, centroid_ids, pids = solve_kmedoids(
                X, pivot_indices, pivot_scores, k
            )
        except Exception as e:
            print(f"ERROR in solve_kmedoids: {e}")
            continue

        # Assign clusters
        try:
            clusters, labels = assign_clusters(
                X, centroid_vectors, centroid_ids, doc_ids
            )
        except Exception as e:
            print(f"ERROR in assign_clusters: {e}")
            continue

        # Evaluate
        try:
            dbi  = evaluate_dbi(X, labels)
            print(f"\n  Davies-Bouldin Index : {dbi:.6f}  (lower = better)")
        except Exception as e:
            print(f"  DBI error: {e}")
            dbi = None

        try:
            ndcg = evaluate_ndcg(X, clusters, query_df, centroid_ids, doc_ids)
            print(f"  nDCG@10              : {ndcg:.6f}  (higher = better)")
        except Exception as e:
            print(f"  nDCG error: {e}")
            ndcg = None

        # Write submission
        try:
            write_submission(k, centroid_vectors, centroid_ids,
                             clusters, pids)
        except Exception as e:
            print(f"  Submission write error: {e}")

        print(f"\n  k={k} done in {time.time()-t0:.1f}s")

    print(f"\n{'='*60}")
    print(f"ALL DONE")
    print(f"Run again with USE_QPU=True for QPU submission.")
    print(f"Both SA and QPU files needed for valid submission.")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()


Task 3A | QA | FPS-QUERYAWARE
k values: [10, 25, 50] | FPS pivots: 150
Loading documents...
  Docs: 6486 × 768
Loading queries...
  Queries: 50 | Relevance judgements: 1651

[Signals]
  Computing relevance-weighted query signal...
    Query relevance: mean=0.439  max=1.000
  Computing local density...
    Density: mean=0.384  max=1.000

[FPS Pivot Reduction]

  Running query-biased FPS: 6486 docs → 150 pivots...
    FPS step 30/150...
    FPS step 60/150...
    FPS step 90/150...
    FPS step 120/150...
    FPS step 150/150...
    FPS done: 150 pivots in 25.0s
  Pivot score: mean=0.420  max=1.000

k = 10

  Building QUBO for k=10...
    QUBO: 150 vars | 1425 off-diag edges | constraint strength=7.44
  Solving via QPU...
  Solved in 379.0s | pid=38dad065...
  QUBO returned 37 ≠ 10 → score-based correction
  Final centroids: 10
  Cluster sizes: min=304 max=923 mean=649

  Davies-Bouldin Index : 7.062287  (lower = better)
  nDCG@10              : 0.889197  (higher = better)
  Submission 